# Setup

In [1]:
# Standard library imports
import warnings
import os
import math
import copy
from dataclasses import dataclass
from random import shuffle

# Third-party imports: core data handling
import numpy as np
import pandas as pd

# Third-party imports: modeling
import scipy.optimize as opt
from scipy.stats import norm, pearsonr
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cluster import KMeans
from gensim.models import Word2Vec

# Third-party imports: visualization
import matplotlib.pyplot as plt

# Configuration & Settings
warnings.simplefilter(action='ignore', category=FutureWarning)

Let's break down the imports step-by-step:

The **standard library** block brings in `warnings` (so we can silence `FutureWarning` messages from pandas and scikit-learn), `os` for file paths, `math` for scalar special functions — we will need `math.lgamma`, the log-gamma function, when fitting the Beta–Binomial layer of TSB-HB — `copy` and `shuffle` for the cat2vec helper in the final section, and `dataclass`, a decorator that lets us define a small typed container for the fitted TSB-HB parameters without writing boilerplate.

The **core data** block imports `numpy` under the conventional alias `np` (fast array math) and `pandas` as `pd` (labeled DataFrames, which we use for everything tabular).

The **modeling** block is where the domain-specific machinery lives. `scipy.optimize` (aliased `opt`) provides the L-BFGS-B numerical optimizer we use to maximize the Beta–Binomial marginal likelihood and the REML criterion in TSB-HB. From `scipy.stats` we take `norm` (the normal distribution, used for Monte Carlo sampling of log-demand sizes) and `pearsonr` (to quantify shrinkage later). `lightgbm` is the gradient boosting library used in the at-scale section. From scikit-learn we take `LabelEncoder` (integer-encode categorical columns for LightGBM), `Ridge` and `MultiOutputRegressor` (a simple multi-output baseline for new launches), and `KMeans` (clustering products by embedding). `Word2Vec` from `gensim` powers the cat2vec trick for embedding categorical product attributes.

Finally, `matplotlib.pyplot` is imported as `plt` for all plotting, and the configuration line tells the `warnings` module to ignore `FutureWarning`s — they are about *upcoming* API changes and would otherwise clutter our output.

In [ ]:
# general settings
class CFG:
    data_folder = './data/'
    graph_folder = './graphs/'
    img_dim1 = 20
    img_dim2 = 10
    SEED = 42
    metric = 'rmse'
    horizon = 28          # length of the holdout window, in days

# display style
plt.style.use('fivethirtyeight')
plt.rcParams['figure.figsize'] = (CFG.img_dim1, CFG.img_dim2)

np.random.seed(CFG.SEED)

We gather every knob the notebook depends on into a single `CFG` class, so that changing a path or a figure size requires editing exactly one place:

- `data_folder` points at the [M5 Forecasting competition data](https://www.kaggle.com/c/m5-forecasting-accuracy) — daily Walmart sales, the workhorse dataset of this notebook. `launch_folder` points at the (preprocessed) VISUELLE fashion dataset used in the new-launches section.
- `img_dim1` / `img_dim2` set a wide 20×10 default figure size — intermittent series are long and spiky, so they need horizontal room.
- `SEED = 42` is fed to `np.random.seed`, which fixes NumPy's random number generator. This matters because TSB-HB's probabilistic forecasts are produced by Monte Carlo sampling, and KMeans uses random initialization — with a fixed seed the notebook produces *identical* numbers on every run, which is essential for reproducibility.
- `metric = 'rmse'` records our headline point-forecast metric, and `horizon = 28` is the length of the chronological holdout — 28 days, matching the M5 competition's forecast horizon.

The two `plt` lines select the `fivethirtyeight` style sheet and apply the default figure size globally, so individual plotting cells stay clean.

# Utils

In [2]:
def forecast_metrics(y_true, y_pred):
    '''Point-forecast metrics: mean error (bias), MAE and RMSE.'''
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'me':   float(np.round(np.nanmean(y_pred - y_true), 4)),
        'mae':  float(np.round(np.nanmean(np.abs(y_pred - y_true)), 4)),
        'rmse': float(np.round(np.sqrt(np.nanmean((y_pred - y_true) ** 2)), 4)),
    }

`forecast_metrics` takes the observed values `y_true` and the forecasts `y_pred`, coerces both into float NumPy arrays with `np.asarray` (so the function accepts pandas Series and plain lists alike), and returns a dictionary of three rounded scalars:

- **ME** (mean error): the average of `y_pred - y_true`. Unlike the other two, errors of opposite sign cancel, so ME measures *systematic bias* — a negative ME means the model under-forecasts on average. Bias matters operationally: a biased demand forecast translates directly into systematic over- or under-stocking.
- **MAE** (mean absolute error): the average magnitude of the error, in the original units (units sold per day).
- **RMSE** (root mean squared error): squares the errors before averaging, so large misses are penalized disproportionately. This is our primary metric (per `CFG.metric`), since in inventory management a few huge errors hurt more than many tiny ones.

We use `np.nanmean` rather than `np.mean` throughout so that any NaN padding in a forecast array is ignored rather than poisoning the result, and we convert to native `float` and round to 4 decimals so the dictionaries print cleanly.